In [ ]:
# Cell 1: Install
!pip install -q ultralytics opencv-python matplotlib

In [ ]:
# Cell 2: Imports
from ultralytics import YOLO
import cv2
import os, glob, shutil, random, collections
import matplotlib.pyplot as plt

In [ ]:
# Cell 3: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4: Prepare data + data.yaml (run once; re-running rebuilds everything cleanly)
BASE  = "/content/drive/MyDrive/dog_cat_dataset/dog_cat"
LOCAL = "/content/dog_cat"

# Copy dataset to local disk (faster than Drive)
shutil.rmtree(LOCAL, ignore_errors=True)
shutil.copytree(BASE, LOCAL, ignore=shutil.ignore_patterns("*.cache", "data.yaml"))

# Make a 15% valid split from train
os.makedirs(f"{LOCAL}/valid/images", exist_ok=True)
os.makedirs(f"{LOCAL}/valid/labels", exist_ok=True)
random.seed(42)
imgs = sorted(glob.glob(f"{LOCAL}/train/images/*"))
for p in random.sample(imgs, int(0.15 * len(imgs))):
    stem = os.path.splitext(os.path.basename(p))[0]
    shutil.move(p, f"{LOCAL}/valid/images/")
    lbl = f"{LOCAL}/train/labels/{stem}.txt"
    if os.path.exists(lbl):
        shutil.move(lbl, f"{LOCAL}/valid/labels/")

data_yaml = f"""
path: {LOCAL}
train: train/images
val: valid/images

nc: 2
names: ['cat', 'dog']
"""
with open(f"{LOCAL}/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml file created successfully")
print("train labels:", len(glob.glob(f"{LOCAL}/train/labels/*.txt")))   # must be above 0
print("valid labels:", len(glob.glob(f"{LOCAL}/valid/labels/*.txt")))

In [ ]:
# Cell 5: Sanity-check labels (class 0 = cat, class 1 = dog)
for split in ["train", "valid"]:
    c = collections.Counter()
    empty = 0
    for f in glob.glob(f"{LOCAL}/{split}/labels/*.txt"):
        lines = [l.split() for l in open(f).read().strip().splitlines() if l.strip()]
        if not lines:
            empty += 1
        for parts in lines:
            c[parts[0]] += 1          # count EVERY box, not just the first line
    print(f"{split}: cat(0)={c['0']} | dog(1)={c['1']} | other={sum(v for k, v in c.items() if k not in ('0','1'))} | empty label files={empty}")

print("filenames with 'dog':", len([p for p in os.listdir(f"{BASE}/train/images") if "dog" in p.lower()]))
print("filenames with 'cat':", len([p for p in os.listdir(f"{BASE}/train/images") if "cat" in p.lower()]))

In [ ]:
# Cell 6: Train
model = YOLO("yolov8n.pt")

model.train(
    data=f"{LOCAL}/data.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    name="dog_cat_detector"
)

In [ ]:
# Cell 7: Load the trained model and save it to Drive
weights = glob.glob("/content/runs/**/best.pt", recursive=True)
if not weights:
    weights = glob.glob("/content/runs/**/last.pt", recursive=True)

# If Colab was restarted, fall back to the copy saved on Drive
drive_copy = "/content/drive/MyDrive/best.pt"
if not weights and os.path.exists(drive_copy):
    weights = [drive_copy]

if not weights:
    print("No trained weights found. Cell 6 (Train) did not finish.")
    print("Fix: re-run cell 4, then cell 6 and wait for 'Results saved to ...'")
else:
    best = max(weights, key=os.path.getmtime)
    print("using:", best)
    trained_model = YOLO(best)
    if best != drive_copy:
        shutil.copy(best, drive_copy)
        print("saved to Drive:", drive_copy)

In [ ]:
# Cell 8: Show detection images (MAX 10) with a border box around each cat / dog
MAX_IMAGES = 10      # <-- limit
CONF = 0.25          # minimum confidence to draw a box
COLORS = {"cat": (255, 140, 0), "dog": (0, 200, 0)}   # RGB: cat=orange, dog=green

test_folder = f"{LOCAL}/test/images"
if not glob.glob(f"{test_folder}/*"):
    test_folder = f"{LOCAL}/valid/images"

exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
image_files = sorted(f for f in os.listdir(test_folder) if f.lower().endswith(exts))
random.seed(42)
image_files = random.sample(image_files, min(MAX_IMAGES, len(image_files)))

shown = 0
for image_name in image_files:
    bgr = cv2.imread(os.path.join(test_folder, image_name))
    if bgr is None:
        continue
    result = trained_model(bgr, conf=CONF, verbose=False)[0]
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]
    thick = max(3, int(round(0.005 * max(h, w))))          # border thickness
    scale = max(0.6, 0.0012 * max(h, w))                   # text size

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        label = result.names[int(box.cls[0])]
        conf = float(box.conf[0])
        color = COLORS.get(label, (255, 0, 0))

        cv2.rectangle(img, (x1, y1), (x2, y2), color, thick)          # border box
        text = f"{label} {conf:.2f}"
        (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        top = max(y1, th + bl + 4)
        cv2.rectangle(img, (x1, top - th - bl - 4), (x1 + tw + 6, top), color, -1)
        cv2.putText(img, text, (x1 + 3, top - bl - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2)

    shown += 1
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.title(f"[{shown}/{len(image_files)}] {image_name}  |  detections: {len(result.boxes)}")
    plt.axis("off")
    plt.show()

print(f"Displayed {shown} image(s) (limit = {MAX_IMAGES})")